**Group information**

| Family name | First name | Email address |
| ----------- | ---------- | ------------- |
|             |            |               |
|             |            |               |
|             |            |               |

# House price prediction - Solutions

This tutorial explores how to predict housing prices from property images. The objective is to perform image-based regression using the [SoCal](https://www.kaggle.com/datasets/ted8080/house-prices-and-images-socal) dataset, which contains 15,475 observations with house prices, exterior house images, and several housing attributes including surface, the number of bedrooms and bathrooms. 

For this assignment, we adopt a transfer learning approach based on a Vision Transformer (ViT) backbone pre-trained using the Masked Autoencoder (MAE) objective (ViT-MAE) from [He et al. (2022)](https://arxiv.org/abs/2111.06377). The encoder is first pre-trained using masked image reconstruction and then fine-tuned for house-price prediction.

**Note:** `transformers` is a large and complex library, so some environment configuration and debugging may be required to ensure compatibility with PyTorch. The dataset is not fully cleaned and contains several images that do not depict the house facade.

**Large language models** can be powerful learning tools, but make sure to continue asking questions until you fully understand the produced answer and can judge its correctness. Simply copy-pasting generated code does not contribute to learning and is a waste of your class time.

In [ ]:
# Google Colab
!pip install lightning, transformers

In [ ]:
# Packages
import lightning
import os
import pandas as pd
import shutil
import torch
import transformers
import tqdm

from matplotlib import pyplot as plt
from lightning.pytorch import callbacks, loggers
from torch import nn, optim, utils
from torchvision import io, ops
from urllib import request

# Device
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

In [ ]:
# Utilities
def download_data():
    if os.getcwd().endswith('/data'):
        print('Data folder already exists')
    else:
        request.urlretrieve('https://www.dropbox.com/scl/fi/lp1wttt5ejygsxr146t0r/housing_data.zip?rlkey=bhd682ybazq1dddze8z0k2oi6&dl=1', 'data.zip')
        shutil.unpack_archive('data.zip', 'data')
        os.remove('data.zip')
        os.chdir('data')

def display_image(image:torch.Tensor, title:str='', cmap:str='gray', figsize=(5, 5)) -> None:
    image   = torch.einsum('dhw -> hwd', image)
    fig, ax = plt.subplots(1, figsize=figsize)
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=15)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()
    plt.close()

def unprocess_image(image:torch.Tensor, processor:nn.Module) -> torch.Tensor:
    mean  = torch.Tensor(processor.image_mean).view(3, 1, 1)
    std   = torch.Tensor(processor.image_std).view(3, 1, 1)
    image = (image * std + mean) * 255
    image = torch.clip(image, 0, 255).to(torch.uint8)
    return image

In [ ]:
# Downloads dataset
os.chdir(os.path.expanduser('~/Library/CloudStorage/Dropbox/teaching/image_analysis/5_vision_transformers/vision_transformers_practice/housing_data'))
# download_data()

**1. Backbone** 

Load the pre-trained ViT-MAE backbone with weights from `facebook/vit-mae-base` and its corresponding Hugging Face image processor. Make sure to load the model using `transformers.ViTMAEForPreTraining`, which includes both the encoder and the decoder. By contrast `transformers.ViTMAEModel` contains only the encoder. 

In [ ]:
# Backbone
backbone  = 'facebook/vit-mae-base'
backbone  = os.path.expanduser('~/.cache/huggingface/hub/models--facebook--vit-mae-base/snapshots/25b184bea5538bf5c4c852c79d221195fdd2778d')
processor = transformers.AutoImageProcessor.from_pretrained(backbone, input_data_format='NCHW')
vitmae_model = transformers.ViTMAEForPreTraining.from_pretrained(backbone)

**2. Data**

Load the images and housing data, convert them to `torch.Tensor` objects. Use the logarithm of the house price per square metre as the target variable, and ensure that all tensors use the `torch.float32` data type. Display a sample of images with their corresponding prices using the provided function.

In [ ]:
## Loads images
data    = pd.read_csv('dataset.csv', index_col='image')
targets = torch.from_numpy(data['price'].values / data['surface'].values)
targets = torch.log(targets).to(torch.float32)
images  = torch.stack([io.read_image(path=f'images/{file}.jpg') for file in data.index], dim=0)

# Displays images and labels
for i in torch.randint(0, len(images), (1,)):
    display_image(images[i], title=f'Log price per square unit: ${targets[i]:.2f}')

**3. Data module**

Implement a PyTorch `LightningDataModule` that splits the image dataset into training, validation, and test sets, applies the Hugging Face image processor during batching, and returns appropriately configured `dataloaders` for model training and evaluation. Test the `dataloader` by loading a sample batch.

In [ ]:
class VitMAEDataModule(lightning.LightningDataModule):

    def __init__(self, images:torch.Tensor, processor, batch_size:int=32, splits=(0.70, 0.15, 0.15)):
        super().__init__()
        self.images     = images
        self.processor  = processor
        self.batch_size = batch_size
        self.splits     = splits

    def setup(self, stage:str=None):
        self.train_dataset, self.valid_dataset, self.test_dataset = torch.utils.data.random_split(self.images, self.splits)

    def collate(self, batch:list) -> torch.Tensor:
        return self.processor(batch, return_tensors='pt')['pixel_values']

    def train_dataloader(self):
        return torch.utils.data.DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True,  collate_fn=self.collate)

    def val_dataloader(self):
        return torch.utils.data.DataLoader(self.valid_dataset, batch_size=self.batch_size, shuffle=False, collate_fn=self.collate)

    def test_dataloader(self):
        return torch.utils.data.DataLoader(self.test_dataset,  batch_size=self.batch_size, shuffle=False, collate_fn=self.collate)

''' Tests dataloader
data_module = VitMAEDataModule(images, processor, batch_size=64)
data_module.setup()
batch = next(iter(data_module.train_dataloader()))
del data_module, batch
'''

**4. Model module**

Implement a PyTorch `LightningModule` for pre-training the ViT-MAE model. The module should define the forward pass, training and validation steps using the masked reconstruction loss, and configure the AdamW optimiser with a reasonable learning rate and weight decay.

In [ ]:
class VitMAEModelModule(lightning.LightningModule):

    def __init__(self, model, learning_rate:float, weight_decay:float):
        super().__init__()
        self.save_hyperparameters(ignore=['model'])
        self.model         = model
        self.learning_rate = learning_rate
        self.weight_decay  = weight_decay

    def forward(self, X:torch.Tensor):
        return self.model(X)

    def _step(self, batch:torch.Tensor, stage:str) -> torch.Tensor:
        loss = self.model(batch).loss
        self.log(f'{stage}_loss', loss, prog_bar=True, sync_dist=True)
        return loss

    def training_step(self, batch:torch.Tensor, batch_idx:int) -> torch.Tensor:
        return self._step(batch, 'train')

    def validation_step(self, batch:torch.Tensor, batch_idx:int) -> torch.Tensor:
        return self._step(batch, 'val')

    def configure_optimizers(self):
        return optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)

**5. Pre-training**

Initialise the data and model modules, then configure a PyTorch Lightning trainer to pre-train the ViT-MAE model. Choose an appropriate batch size. You may also use the `EarlyStopping` and `ModelCheckpointing` callbacks. Train the model and save the training logs and best-performing checkpoint. For this exercise, you can run a single epoch.

Depending on your hardware constraints, you may wish to train the model using `torch.float16` precision to reduce memory usage, at the expense of reduced numerical precision.

In [ ]:
# Initialises data and model modules
data_module = VitMAEDataModule(
    images=images, 
    processor=processor, 
    batch_size=32)

model_module = VitMAEModelModule(
    model=vitmae_model,
    learning_rate=1e-4,
    weight_decay=0.05)

# Initialises trainer
os.makedirs('models', exist_ok=True)

early_stopping   = callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=3)

model_checkpoint = callbacks.ModelCheckpoint(
    dirpath='models', 
    filename='model_pretrained',
    monitor='val_loss')

trainer = lightning.Trainer(
    max_epochs=1, #! For the practice
    accelerator=device,
    log_every_n_steps=10,
    callbacks=[early_stopping, model_checkpoint],
    logger=loggers.CSVLogger(save_dir='models/logs', name='model_pretrained'),
)

# Trains model
trainer.fit(
    model=model_module, 
    datamodule=data_module)

**7. Reconstruction visualisation**

Select a sample image from the dataset and use the pre-trained ViT-MAE model to reconstruct the masked image patches. Visualise the original image, the masked input, the predicted patches, and the final reconstructed image. Comment on the reconstruction quality and the model's ability to recover missing visual information.

In [ ]:
# Predicts image
image  = images[torch.randint(0, len(images), (1,))]
inputs = processor(image, return_tensors='pt').pixel_values
with torch.no_grad():
    outputs = vitmae_model(inputs)

# Unpack results
mask = outputs.mask.unsqueeze(-1).repeat(1, 1, vitmae_model.config.patch_size**2 * 3)
mask = vitmae_model.unpatchify(mask)
pred = vitmae_model.unpatchify(outputs.logits)

# Figures data
figdata = {
    'image':inputs, 
    'masked tiles':inputs * (1 - mask),
    'predicted tiles':pred * mask, 
    'reconstructed image':inputs * (1 - mask) + pred * mask
}

# Figures
fig, axs = plt.subplots(1, 4, figsize=(20, 5))
for ax, (title, image) in zip(axs.ravel(), figdata.items()):
    image = unprocess_image(image, processor)
    image = torch.einsum('dhw->hwd', image.squeeze(0))
    ax.imshow(image)
    ax.set_title(title.capitalize(), fontsize=20)
    ax.axis('off')
plt.tight_layout()
plt.show()
plt.close()

**8. Price prediction model**

Implement a house-price prediction model using the pre-trained ViT-MAE encoder as a feature extractor. Construct a regression head that maps the encoder representations to a continuous house-price prediction, and initialise the model using the pre-trained backbone weights and include only the encoder.

In [ ]:
class price_model(nn.Module):
    
    def __init__(self, encoder:nn.Module, hidden_size:int=768, output_size:int=1):
        super().__init__()
        self.encoder = encoder
        self.output  = nn.Linear(hidden_size, output_size)
    
    def forward(self, X:torch.Tensor) -> torch.Tensor:
        H = self.encoder(X)
        H = H.last_hidden_state[:, 0] # Special token
        Y = self.output(H)
        return Y
    
encoder = VitMAEModelModule.load_from_checkpoint(
    #checkpoint_path=model_checkpoint.best_model_path
    checkpoint_path='models/model_pretrained.ckpt',
    model=vitmae_model,
    learning_rate=1e-4,
    weight_decay=0.05
).model.vit
# encoder = transformers.ViTMAEModel.from_pretrained(backbone)

price_model = price_model(encoder=encoder)

**9. Data and model modules**

Rewrite the `LightningDataModule` to support image–target batches for regression, and implement a `LightningModule` defining the training, validation, and test steps using mean squared error loss.

In [ ]:
class PriceDataModule(lightning.LightningDataModule):

    def __init__(self, images:torch.Tensor, targets:torch.Tensor, processor, batch_size:int=32, splits=(0.70, 0.15, 0.15)):
        super().__init__()
        self.images     = images
        self.targets    = targets
        self.processor  = processor
        self.batch_size = batch_size
        self.splits     = splits

    def setup(self, stage:str=None):
        dataset = torch.utils.data.TensorDataset(self.images, self.targets)
        self.train_dataset, self.valid_dataset, self.test_dataset = torch.utils.data.random_split(dataset, self.splits)

    def collate(self, batch:list) -> tuple:
        images  = self.processor([i[0] for i in batch], return_tensors='pt')['pixel_values']
        targets = torch.stack([i[1] for i in batch])
        return images, targets

    def train_dataloader(self):
        return torch.utils.data.DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True,  collate_fn=self.collate)

    def val_dataloader(self):
        return torch.utils.data.DataLoader(self.valid_dataset, batch_size=self.batch_size, shuffle=False, collate_fn=self.collate)

    def test_dataloader(self):
        return torch.utils.data.DataLoader(self.test_dataset,  batch_size=self.batch_size, shuffle=False, collate_fn=self.collate)

''' Tests dataloader
data_module = PriceDataModule(images, targets, processor, batch_size=64)
data_module.setup()
images, targets = next(iter(data_module.train_dataloader()))
del data_module, images, targets
'''

In [ ]:
class price_modelModule(lightning.LightningModule):

    def __init__(self, model:nn.Module, learning_rate:float, weight_decay:float):
        super().__init__()
        self.save_hyperparameters()
        self.model         = model
        self.criterion     = nn.MSELoss()
        self.learning_rate = learning_rate
        self.weight_decay  = weight_decay
        self.trainable     = None

    def forward(self, X:torch.Tensor) -> torch.Tensor:
        return self.model(X)

    def _step(self, batch:tuple, stage:str) -> torch.Tensor:
        X, Y = batch
        loss = self.criterion(self.model(X).squeeze(), Y)
        self.log(f'{stage}_loss', loss, prog_bar=True, sync_dist=True)
        return loss

    def training_step(self, batch:tuple, batch_idx:int) -> torch.Tensor:
        return self._step(batch, 'train')

    def validation_step(self, batch:tuple, batch_idx:int) -> torch.Tensor:
        return self._step(batch, 'val')

    def test_step(self, batch:tuple, batch_idx:int) -> torch.Tensor:
        return self._step(batch, 'test')

    def configure_optimizers(self):
        return optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
    
    def count_parameters(self):
        trainable = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        nontrain  = sum(p.numel() for p in self.model.parameters() if not p.requires_grad)
        print(f'- Trainable parameters: {trainable:,}\n- Non-trainable parameters: {nontrain:,}')
 
    def freeze_encoder(self):
        self.trainable = [param.requires_grad for param in self.model.encoder.parameters()]
        for param in self.model.encoder.parameters():
            param.requires_grad = False
        print('Encoder frozen')
        self.count_parameters()

    def unfreeze_encoder(self, n_layers=None):
        for param in self.model.encoder.parameters():
            param.requires_grad = n_layers is None
        if n_layers is not None:
            for layer in self.model.encoder.encoder.layer[-n_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True
        for param in self.model.output.parameters():
            param.requires_grad = True
        self.trainer.strategy.setup_optimizers(self.trainer)
        print(f'Last {n_layers} layers of the encoder unfrozen, optimisers reset')
        self.count_parameters()

**10. Fine-Tuning**

Configure a PyTorch Lightning `Trainer` for house-price prediction, including optional early stopping and model checkpointing. First train only the regression head with the encoder frozen, then unfreeze the encoder and fine-tune the entire model using a smaller learning rate.

In [ ]:
# Initialises data and model modules
data_module = PriceDataModule(
    images=images,
    targets=targets,
    processor=processor, 
    batch_size=32)

model_module = price_modelModule(
    model=price_model,
    learning_rate=1e-4,
    weight_decay=0.05)

# Initialises trainer
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=3)

model_checkpoint = callbacks.ModelCheckpoint(
    dirpath='models', 
    filename='model_finetuned',
    monitor='val_loss')

trainer = lightning.Trainer(
    max_epochs=100,
    accelerator=device,
    log_every_n_steps=10,
    callbacks=[early_stopping, model_checkpoint],
    logger=loggers.CSVLogger(save_dir='models/logs', name='model_finetuned')
)

# Aligns classification head
model_module.learning_rate = 1e-4
model_module.freeze_encoder()
trainer.fit(model_module, datamodule=data_module)

# Fine-tunes entire model
model_module.learning_rate = 1e-5
model_module.unfreeze_encoder(n_layers=1)
trainer.fit(model_module, datamodule=data_module) # ckpt_path=f'{paths.models}/vitmae_finetuned.ckpt'


**Bonus**

Modify the price prediction model and the data loader to incorporate additional covariates alongside the image inputs, such as the standardised surface area, number of rooms, number of bathrooms, and location information.